In [65]:
import os
import random
import numpy as np
import pandas as pd

from gensim.models import FastText
from gensim.utils import simple_preprocess

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

DATA_DIR = "../data" 
OUT_DIR = "embeddings"
os.makedirs(OUT_DIR, exist_ok=True)

In [66]:
def load_dataset(path: str, text_col: str = "embedding_text") -> pd.DataFrame:
    df = pd.read_csv(path)
    if text_col not in df.columns:
        raise ValueError(
            f"Column '{text_col}' not found in {path}. "
            f"Available columns: {list(df.columns)}"
        )
    df[text_col] = df[text_col].fillna("").astype(str)
    return df


import re

def make_sentences(df: pd.DataFrame, text_col: str = "embedding_text"):
    sentences = []
    for text in df[text_col].tolist():
        if not isinstance(text, str) or not text.strip():
            continue
        # Lowercase, but preserve tokens with digits/dots/plus (c++, node.js, k8s)
        text = text.lower()
        tokens = re.findall(r'\b[a-z][a-z0-9\.+#_-]{1,}\b', text)
        tokens = [t for t in tokens if len(t) >= 2]
        if tokens:
            sentences.append(tokens)
    return sentences


def corpus_stats(sentences):
    n_docs = len(sentences)
    n_tokens = sum(len(s) for s in sentences)
    avg_len = n_tokens / n_docs if n_docs else 0
    vocab = set(tok for s in sentences for tok in s)
    return {
        "docs": n_docs,
        "tokens": n_tokens,
        "avg_doc_len": avg_len,
        "vocab_size": len(vocab),
    }

In [67]:
group_files = {
    "entry": os.path.join(DATA_DIR, "job_postings_tech_entry.csv"),
    "mid": os.path.join(DATA_DIR, "job_postings_tech_mid.csv"),
    "senior": os.path.join(DATA_DIR, "job_postings_tech_senior.csv"),
    "leadership": os.path.join(DATA_DIR, "job_postings_tech_leadership.csv"),
}

TEXT_COL = "embedding_text"  

In [68]:
def train_fasttext(
    sentences,
    vector_size=200,
    window=5,
    min_count=5,
    workers=4,
    epochs=15,
    sg=1,          # 1 = skip-gram 
    negative=10,
    min_n=3,       # subword min length
    max_n=6,       # subword max length
):
    """
    fastText-style embeddings via gensim FastText.
    - sg=1 => skip-gram
    - min_n/max_n => character n-grams (subword info)
    """
    model = FastText(
        vector_size=vector_size,
        window=window,
        min_count=min_count,
        workers=workers,
        sg=sg,
        negative=negative,
        min_n=min_n,
        max_n=max_n,
        seed=SEED,
    )

    # Build vocab
    model.build_vocab(corpus_iterable=sentences)
    print(f"Built vocab size: {len(model.wv)}")

    # Train
    model.train(
        corpus_iterable=sentences,
        total_examples=len(sentences),
        epochs=epochs,
    )
    return model

In [69]:
for group, path in group_files.items():
    df_check = pd.read_csv(path)
    print("\n" + "=" * 60)
    print("GROUP:", group)
    print("Columns:", df_check.columns.tolist())
    print(df_check[TEXT_COL].head(2).tolist())

    tokens = set(" ".join(df_check[TEXT_COL].fillna("").astype(str)).split())
    for bad in ["male", "female", "both", "years"]:
        print(f"{bad} in tokens? ->", bad in tokens)


GROUP: entry
Columns: ['title', 'description', 'experience', 'qualifications', 'responsibilities', 'preference', 'text', 'clean_text', 'embedding_text', 'seniority']
['protect an organizations computer networks and systems from security threats monitor network traffic and respond to incidents. manage and secure computer networks including firewalls and intrusion detection systems. monitor network traffic for security threats. implement security policies and procedures.', 'soc analysts work in security operations centers. they monitor and respond to security incidents analyze threats and implement security protocols to protect an organizations assets. work in soc teams to monitor and respond to security incidents. investigate and analyze security alerts and incidents. maintain security documentation and incident reports.']
male in tokens? -> False
female in tokens? -> False
both in tokens? -> False
years in tokens? -> False

GROUP: mid
Columns: ['title', 'description', 'experience', 'q

In [70]:
def get_min_count(stats):
    docs = stats["docs"]
    if docs < 500:
        return 1
    elif docs < 2000:
        return 2
    elif docs < 5000:
        return 3
    else:
        return 5
    
trained_models = {}

for group, path in group_files.items():
    print("\n" + "="*80)
    print(f"GROUP: {group} | FILE: {path}")

    df = load_dataset(path, text_col=TEXT_COL)
    sentences = make_sentences(df, text_col=TEXT_COL)

    stats = corpus_stats(sentences)
    print("Corpus stats:", stats)

    min_count = get_min_count(stats)
    print(f"Using min_count={min_count} for {stats['docs']} docs")

    if stats["vocab_size"] < 5000:
        min_count = max(1, min_count - 1)
        print(f"Small vocab detected -> lowering min_count to {min_count}")

    model = train_fasttext(
        sentences,
        vector_size=100,
        window=5,
        min_count=min_count,
        workers=4,
        epochs=15,
        sg=1,
        negative=5,
        min_n=3,
        max_n=5,
        )

    # Save model + vectors
    model_path = os.path.join(OUT_DIR, f"fasttext_{group}.model")
    vec_path = os.path.join(OUT_DIR, f"fasttext_{group}.vectors.kv")

    model.save(model_path)
    model.wv.save(vec_path)

    print(f"Saved model to: {model_path}")
    print(f"Saved vectors to: {vec_path}")

    trained_models[group] = model


GROUP: entry | FILE: ../data/job_postings_tech_entry.csv
Corpus stats: {'docs': 80, 'tokens': 3869, 'avg_doc_len': 48.3625, 'vocab_size': 683}
Using min_count=1 for 80 docs
Small vocab detected -> lowering min_count to 1
Built vocab size: 683
Saved model to: embeddings/fasttext_entry.model
Saved vectors to: embeddings/fasttext_entry.vectors.kv

GROUP: mid | FILE: ../data/job_postings_tech_mid.csv
Corpus stats: {'docs': 68, 'tokens': 3339, 'avg_doc_len': 49.10294117647059, 'vocab_size': 616}
Using min_count=1 for 68 docs
Small vocab detected -> lowering min_count to 1
Built vocab size: 616
Saved model to: embeddings/fasttext_mid.model
Saved vectors to: embeddings/fasttext_mid.vectors.kv

GROUP: senior | FILE: ../data/job_postings_tech_senior.csv
Corpus stats: {'docs': 72, 'tokens': 3535, 'avg_doc_len': 49.09722222222222, 'vocab_size': 632}
Using min_count=1 for 72 docs
Small vocab detected -> lowering min_count to 1
Built vocab size: 632
Saved model to: embeddings/fasttext_senior.model

In [71]:
print(model.wv.index_to_key[:50])

['and', 'architectural', 'for', 'the', 'design', 'with', 'designs', 'it', 'infrastructure', 'technology', 'solutions', 'on', 'to', 'ensure', 'quality', 'qa', 'architect', 'building', 'practices', 'projects', 'organizations', 'an', 'including', 'lead', 'standards', 'processes', 'manage', 'develop', 'implement', 'aligning', 'security', 'scalability', 'cloud', 'overall', 'enterprise', 'software', 'business', 'develops', 'solution', 'structures', 'buildings', 'plans', 'create', 'focus', 'into', 'eco-friendly', 'sustainable', 'regulations', 'ensuring', 'budgets']


In [72]:
print("TEXT_COL =", TEXT_COL)
print(df[TEXT_COL].head(5).tolist())

TEXT_COL = embedding_text
['a cloud architect designs and manages cloud-based solutions optimizing scalability security and performance while aligning them with the companys technology strategy. design and implement cloud-based solutions optimizing for performance and cost-efficiency. develop cloud migration strategies. architect and manage cloud infrastructure.', 'a qa manager is responsible for leading and managing the quality assurance team. they ensure product quality develop and implement qa processes and maintain quality standards. lead and manage the qa team including test planning resource allocation and test strategy development. define quality standards and metrics. oversee qa processes and continuous improvement efforts.', 'an infrastructure manager is responsible for the operation and maintenance of an organizations it infrastructure including servers networks and data centers. manage it infrastructure including servers networks and data centers. ensure high availability se

In [73]:
def check_word(model, word):
    if word in model.wv:
        return model.wv.most_similar(word, topn=10)
    return None

for group, model in trained_models.items():
    print("\n" + "-"*80)
    print("Group:", group)

    for w in ["leadership", "manager", "collaboration", "python", "nurturing"]:
        sims = check_word(model, w)
        if sims:
            print(f"Most similar to '{w}':", sims[:5])
        else:
            print(f"'{w}' not in vocab")


--------------------------------------------------------------------------------
Group: entry
Most similar to 'leadership': [('architectural', 0.9999312162399292), ('understand', 0.999930202960968), ('oversees', 0.9999285340309143), ('consistency', 0.9999276399612427), ('critical', 0.9999261498451233)]
Most similar to 'manager': [('managers', 0.9999604821205139), ('manage', 0.9999560117721558), ('manages', 0.9999505877494812), ('infrastructure', 0.9999335408210754), ('management', 0.9999263882637024)]
Most similar to 'collaboration': [('relational', 0.9999595880508423), ('non-relational', 0.9999595880508423), ('specifications', 0.9999558925628662), ('operation', 0.9999545216560364), ('presentation', 0.9999544620513916)]
Most similar to 'python': [('timelines', 0.9999078512191772), ('presentation', 0.9998937249183655), ('landscape', 0.9998907446861267), ('upgrades', 0.99988853931427), ('backups', 0.9998880624771118)]
Most similar to 'nurturing': [('considering', 0.9999229907989502), ('

In [74]:
import pandas as pd
from collections import Counter

df = pd.read_csv("../data/job_postings_tech_entry.csv")

TEXT_COL = "embedding_text"   # or "processed_text" if that's what you're using
print("Sample rows:")
for i in range(5):
    print("-", df[TEXT_COL].iloc[i])

# Approx unique token count (simple)
tokens = " ".join(df[TEXT_COL].fillna("").astype(str)).split()
print("Approx unique tokens:", len(set(tokens)))
print("Top 30 tokens:", Counter(tokens).most_common(30))

Sample rows:
- protect an organizations computer networks and systems from security threats monitor network traffic and respond to incidents. manage and secure computer networks including firewalls and intrusion detection systems. monitor network traffic for security threats. implement security policies and procedures.
- soc analysts work in security operations centers. they monitor and respond to security incidents analyze threats and implement security protocols to protect an organizations assets. work in soc teams to monitor and respond to security incidents. investigate and analyze security alerts and incidents. maintain security documentation and incident reports.
- user interface designers focus on the visual and interactive aspects of digital interfaces. they design layouts buttons and other elements to ensure a cohesive and visually appealing user interface. create visually appealing user interfaces ui that align with brand aesthetics and enhance user engagement. design interfa